In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.trafa_t10026

In [0]:
%sql
SELECT regkom, ar,
       SUM(CASE WHEN drivmedel <> 't1' THEN CAST(itrfslut AS BIGINT) END) AS sum_fuels,
       MAX(CASE WHEN drivmedel =  't1' THEN CAST(itrfslut AS BIGINT) END) AS total_t1,
       MAX(CASE WHEN drivmedel = '105' THEN CAST(itrfslut AS BIGINT) END) AS phev
FROM laddstolpar_df.bronze.trafa_t10026
WHERE regkom IN ('0180', '2584') AND ar = '2025'
GROUP BY regkom, ar;

In [0]:
%sql DESCRIBE TABLE laddstolpar_df.bronze.elpris

In [0]:
%sql
SELECT COUNT(*)                               AS silver_rows,
       (SELECT COUNT(*) FROM laddstolpar_df.bronze.elpris) AS bronze_rows,
       MIN(time_start_utc), MAX(time_start_utc),
       SUM(CASE WHEN sek_per_kwh < 0 THEN 1 END) AS negative,
       MAX(length(split(CAST((SELECT MAX(SEK_per_kWh) FROM laddstolpar_df.bronze.elpris) AS STRING), '\\.')[1])) AS max_decimals_sample
FROM laddstolpar_df.silver.elpris;

In [0]:
%sql
SELECT elomrade, datum_lokal, time_start_utc, time_end_utc,
       unix_timestamp(time_end_utc) - unix_timestamp(time_start_utc) AS seconds,
       sek_per_kwh, _file
FROM laddstolpar_df.silver.elpris
WHERE unix_timestamp(time_end_utc) - unix_timestamp(time_start_utc) <> 900
   OR sek_per_kwh NOT BETWEEN -10 AND 50
ORDER BY time_start_utc, elomrade;

In [0]:
%sql
SELECT elomrade, time_start, time_end
FROM laddstolpar_df.bronze.elpris
WHERE elomrade = 'SE3'
  AND time_start BETWEEN '2025-10-26T02:30' AND '2025-10-26T03:15'
ORDER BY _file, time_start;

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(datum_lokal) AS with_date,
       MIN(datum_lokal), MAX(datum_lokal)
FROM laddstolpar_df.silver.elpris;

In [0]:
%sql
SELECT expected_rows, COUNT(*) AS zone_days,
       SUM(CASE WHEN n_rows = expected_rows THEN 1 ELSE 0 END) AS complete,
       SUM(CASE WHEN n_rows <> n_distinct_starts THEN 1 ELSE 0 END) AS with_duplicates
FROM laddstolpar_df.ops.elpris_day_check
GROUP BY expected_rows
ORDER BY expected_rows;

In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.seed_skr_kommungrupp;
DESCRIBE TABLE laddstolpar_df.bronze.seed_elomrade_lan_default;
DESCRIBE TABLE laddstolpar_df.bronze.seed_elomrade_kommun_override;

In [0]:
%sql
-- 1. Shape: expect 290 / 290 / 79 override / 211 county default
SELECT COUNT(*) AS rows, COUNT(DISTINCT kommun_kod) AS kommuner,
       SUM(CASE WHEN elomrade_level = 'kommun_override' THEN 1 ELSE 0 END) AS override,
       SUM(CASE WHEN elomrade_level = 'lan_default'     THEN 1 ELSE 0 END) AS lan_default,
       SUM(CASE WHEN elomrade_is_split     THEN 1 ELSE 0 END) AS split,          -- expect 13
       SUM(CASE WHEN elomrade_needs_review THEN 1 ELSE 0 END) AS needs_review    -- expect 12
FROM laddstolpar_df.silver.kommun;

-- 2. Zone distribution: expect SE1 19, SE2 34, SE3 177, SE4 60 (bronze check 2026-09-24)
SELECT elomrade, COUNT(*) FROM laddstolpar_df.silver.kommun GROUP BY elomrade ORDER BY elomrade;

-- 3. Override codes that did NOT match an SKR kommun (a typo would be silently lost): expect 0 rows
SELECT o.kommun_kod, o.kommun_namn
FROM laddstolpar_df.bronze.seed_elomrade_kommun_override o
LEFT ANTI JOIN laddstolpar_df.silver.kommun k USING (kommun_kod);

-- 4. Name mismatch between seeds (same code, different spelling): expect 0 rows
SELECT k.kommun_kod, k.kommun_namn AS skr_namn, o.kommun_namn AS override_namn
FROM laddstolpar_df.silver.kommun k
JOIN laddstolpar_df.bronze.seed_elomrade_kommun_override o USING (kommun_kod)
WHERE k.kommun_namn <> o.kommun_namn;

In [0]:
%sql
SELECT _file, COUNT(*) AS rows, MAX(_ingested_at) AS loaded
FROM laddstolpar_df.bronze.seed_elomrade_kommun_override
GROUP BY _file;

In [0]:
%sql
-- a) The 12 reviewed kommuner, now with their sourced primary
SELECT kommun_kod, kommun_namn, elomrade AS primary_zone, elomrade_secondary, elomrade_basis
FROM laddstolpar_df.silver.kommun
WHERE elomrade_is_split
ORDER BY kommun_kod;

-- b) The zone distribution after the review (before: SE1 19, SE2 34, SE3 177, SE4 60)
SELECT elomrade, COUNT(*) FROM laddstolpar_df.silver.kommun GROUP BY elomrade ORDER BY elomrade;

In [0]:
%sql
SELECT DISTINCT _file FROM laddstolpar_df.bronze.seed_elomrade_kommun_override;

In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.scb_tab3277;

In [0]:
%sql
-- 1. The table: expect 8 rows, 8 distinct SCB codes, 8 distinct Trafikanalys codes, 2 laddbar
SELECT COUNT(*) AS rows, COUNT(DISTINCT scb_kod) AS scb, COUNT(DISTINCT trafa_kod) AS trafa,
       SUM(CASE WHEN laddbar THEN 1 ELSE 0 END) AS laddbar
FROM laddstolpar_df.silver.drivmedel;

-- 2. SCB codes in the source that are NOT mapped: expect 0 rows
SELECT DISTINCT b.drivmedel FROM laddstolpar_df.bronze.scb_tab3277 b
LEFT ANTI JOIN laddstolpar_df.silver.drivmedel d ON d.scb_kod = b.drivmedel;

-- 3. Trafikanalys codes NOT mapped: expect exactly one row, 't1' (the total, dropped by design)
SELECT DISTINCT b.drivmedel, b.drivmedel_label FROM laddstolpar_df.bronze.trafa_t10026 b
LEFT ANTI JOIN laddstolpar_df.silver.drivmedel d ON d.trafa_kod = b.drivmedel;

In [0]:
%sql
SELECT * FROM laddstolpar_df.silver.drivmedel ORDER BY scb_kod

In [0]:
%sql
-- 1. Profile
SELECT
  COUNT(*)                                                        AS rows,
  COUNT(DISTINCT _snapshot_updated)                               AS versions,        -- expect 1 (one SCB publication)
  COUNT(DISTINCT _file)                                           AS files,           -- expect 6 (year slices)
  collect_set(contentscode)                                       AS contentscodes,
  MIN(tid) AS first_period, MAX(tid) AS last_period,
  COUNT(DISTINCT tid)                                             AS periods,         -- expect 68
  COUNT(DISTINCT region)                                          AS regions,         -- expect 315
  COUNT(DISTINCT CASE WHEN length(region) = 4 THEN region END)    AS regions_4digit,  -- expect 291 (incl. 1917)
  collect_set(status)                                             AS statuses,
  SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END)                  AS null_values,
  SUM(CASE WHEN value <> floor(value) THEN 1 ELSE 0 END)          AS non_integer
FROM laddstolpar_df.bronze.scb_tab3277;

-- 2. Key uniqueness per version (AUTO CDC requirement): expect 0
SELECT COUNT(*) AS duplicate_keys FROM (
  SELECT region, drivmedel, tid, contentscode, _snapshot_updated
  FROM laddstolpar_df.bronze.scb_tab3277
  GROUP BY ALL
  HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT DISTINCT _snapshot_updated FROM laddstolpar_df.bronze.scb_tab3277;

In [0]:
%sql
-- 1. Shape: expect 157,760 rows (290 kommuner × 8 fuels × 68 months), all current, 290 kommuner
SELECT COUNT(*) AS rows,
       SUM(CASE WHEN __END_AT IS NULL THEN 1 ELSE 0 END) AS current_rows,
       COUNT(DISTINCT kommun_kod) AS kommuner,
       COUNT(DISTINCT period) AS periods,
       MIN(__START_AT) AS version_from
FROM laddstolpar_df.silver.scb_tab3277;

-- 2. Sanity: BEV new registrations, all Sweden, 2025 (fuel code 120)
SELECT SUM(antal) AS bev_2025
FROM laddstolpar_df.silver.scb_tab3277
WHERE drivmedel_scb_kod = '120' AND period LIKE '2025M%' AND __END_AT IS NULL;

In [0]:
%sql
WITH k AS (
  SELECT SUM(antal) AS kommun_sum
  FROM laddstolpar_df.silver.scb_tab3277
  WHERE drivmedel_scb_kod = '120' AND period LIKE '2025M%' AND __END_AT IS NULL
),
r AS (
  SELECT SUM(value) AS riket
  FROM laddstolpar_df.bronze.scb_tab3277
  WHERE region = '00' AND drivmedel = '120' AND tid LIKE '2025M%'
)
SELECT kommun_sum, riket, kommun_sum - riket AS diff,
       ROUND(100 * (kommun_sum - riket) / riket, 3) AS diff_pct
FROM k CROSS JOIN r;

In [0]:
%sql
DESCRIBE TABLE laddstolpar_df.bronze.scb_tab628;
DESCRIBE TABLE laddstolpar_df.bronze.scb_tab3276;

In [0]:
%sql
-- TAB628: per measure
SELECT contentscode,
       COUNT(*) AS rows,
       collect_set(kon) AS kon,
       MIN(tid) AS first, MAX(tid) AS last, COUNT(DISTINCT tid) AS years,
       COUNT(DISTINCT region) AS regions,
       COUNT(DISTINCT CASE WHEN length(region) = 4 THEN region END) AS regions_4digit,
       COUNT(DISTINCT _snapshot_updated) AS versions,
       collect_set(status) AS statuses,
       SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS nulls,
       SUM(CASE WHEN value <> floor(value) THEN 1 ELSE 0 END) AS non_integer
FROM laddstolpar_df.bronze.scb_tab628
GROUP BY contentscode ORDER BY contentscode;

In [0]:
%sql
-- TAB3276: per owner category and measure
SELECT agarkategori, contentscode,
       COUNT(*) AS rows,
       MIN(tid) AS first, MAX(tid) AS last,
       COUNT(DISTINCT CASE WHEN length(region) = 4 THEN region END) AS regions_4digit,
       COUNT(DISTINCT _snapshot_updated) AS versions,
       collect_set(status) AS statuses,
       SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS nulls,
       SUM(CASE WHEN value IS NULL AND length(region) = 4 AND region <> '1917' THEN 1 ELSE 0 END) AS nulls_in_kommuner,
       SUM(CASE WHEN value <> floor(value) THEN 1 ELSE 0 END) AS non_integer
FROM laddstolpar_df.bronze.scb_tab3276
GROUP BY agarkategori, contentscode ORDER BY agarkategori, contentscode;

In [0]:
%sql
-- Key uniqueness per version, both tables: expect 0 and 0
SELECT 'tab628' AS t, COUNT(*) AS duplicate_keys FROM (
  SELECT region, kon, contentscode, tid, _snapshot_updated FROM laddstolpar_df.bronze.scb_tab628
  GROUP BY ALL HAVING COUNT(*) > 1)
UNION ALL
SELECT 'tab3276', COUNT(*) FROM (
  SELECT region, agarkategori, contentscode, tid, _snapshot_updated FROM laddstolpar_df.bronze.scb_tab3276
  GROUP BY ALL HAVING COUNT(*) > 1);

In [0]:
%sql
-- 1. Shape: expect 5,220 (290 x 3 x 6) and 12,180 (290 x 7 x 6); all rows current
SELECT 'tab628' AS t, COUNT(*) AS rows, SUM(CASE WHEN __END_AT IS NULL THEN 1 ELSE 0 END) AS current_rows,
       COUNT(DISTINCT kommun_kod) AS kommuner, SUM(CASE WHEN varde IS NULL THEN 1 ELSE 0 END) AS nulls
FROM laddstolpar_df.silver.scb_tab628
UNION ALL
SELECT 'tab3276', COUNT(*), SUM(CASE WHEN __END_AT IS NULL THEN 1 ELSE 0 END),
       COUNT(DISTINCT kommun_kod), SUM(CASE WHEN antal IS NULL THEN 1 ELSE 0 END)
FROM laddstolpar_df.silver.scb_tab3276;

-- 2. Reconciliation Σ kommuner vs Riket, 2020 and 2025 (SCB noise starts in 2025, finding 6)
WITH k AS (
  SELECT ar, SUM(varde) AS kommun_sum FROM laddstolpar_df.silver.scb_tab628
  WHERE matt = 'folkmangd' AND ar IN (2020, 2025) AND __END_AT IS NULL GROUP BY ar
), r AS (
  SELECT CAST(tid AS INT) AS ar, SUM(value) AS riket FROM laddstolpar_df.bronze.scb_tab628
  WHERE region = '00' AND contentscode = 'BE0101U2' AND tid IN ('2020', '2025') GROUP BY tid
)
SELECT ar, kommun_sum, riket, kommun_sum - riket AS diff FROM k JOIN r USING (ar) ORDER BY ar;